# Ionosphere Baselines — Persistence, Climatology, OMNI-only GRU, Flat CNN

All four baselines used to evaluate SFNO, trained/defined here and nowhere else:

- **Persistence** — repeat last observed dTEC. No training.
- **Climatology (zero-residual)** — predicts zero residual = the climatological mean. No
  training. (This is *not* Hayden's real per-year climatology `.npz` delivery — that's absolute
  TEC data used to convert predictions back to real TECU, handled entirely in
  `ionosphere_eval.ipynb`. This baseline is purely "predict no anomaly," a residual-space claim
  that needs no external data at all.)
- **OMNI-only GRU** — predicts from solar wind alone, no spatial TEC input. Trained for real
  (an untrained/random-init version is not meaningful evidence of anything).
- **Flat CNN** — parameter-matched (51,182,211 vs. SFNO's 52,316,547) full-map-Conv2d
  counterpart to SFNO. Isolates spherical-harmonic basis vs. flat spatial basis; everything else
  held identical. Not affected by the SFNO filter-shape fix in `ionosphere_s2cnn.ipynb` — Flat
  CNN never had an `l_max`/`m_max` filter to begin with, so its param count was already correct.

**Contains no SFNO code.** `best_model.pt` is never touched here. No storm/quiet or solar-
activity stratification here either — that only matters at eval time on finished checkpoints,
not during training/validation. Belongs entirely in `ionosphere_eval.ipynb`, later.

**Checkpoints produced:** `best_omni_model.pt`, `best_flat_model.pt`, both backed up to
`/content/drive/MyDrive/tec_data/`.

**Training budget: 10 epochs** (vs. SFNO's 20) — both models are far smaller and converge/
overfit faster; matching SFNO's epoch count isn't meaningful here.

## Cell 1 — Setup: mount Drive, copy data locally, install, imports, GPU check

Same pattern as `ionosphere_s2cnn.ipynb`. Skips `README.md.gdoc` (a Google Docs shortcut, not a real file) which broke a plain `shutil.copytree` there.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import shutil, os, json, warnings
from pathlib import Path

DRIVE_DATA_DIR = "/content/drive/MyDrive/tec_data/new_data"
LOCAL_DATA_DIR = "new_data"

os.makedirs(LOCAL_DATA_DIR, exist_ok=True)
for fname in os.listdir(DRIVE_DATA_DIR):
    if fname.endswith('.npy') or fname.endswith('.json'):
        dst = os.path.join(LOCAL_DATA_DIR, fname)
        if not os.path.exists(dst):
            shutil.copy(os.path.join(DRIVE_DATA_DIR, fname), dst)
print("Local data files:", sorted(os.listdir(LOCAL_DATA_DIR)))

!pip install -q torch-harmonics

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import numpy as np
import time
from tqdm.auto import tqdm

from torch_harmonics.quadrature import legendre_gauss_weights

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {DEVICE}')
if DEVICE.type == 'cpu':
    print('WARNING: No GPU detected. Runtime -> Change runtime type -> T4 GPU')

Mounted at /content/drive
Local data files: ['lats.npy', 'lons.npy', 'metadata.json', 'test_omni_input.npy', 'test_target.npy', 'test_tec_input.npy', 'test_window_start_times.npy', 'train_omni_input.npy', 'train_target.npy', 'train_tec_input.npy', 'train_window_start_times.npy', 'val_omni_input.npy', 'val_target.npy', 'val_tec_input.npy', 'val_window_start_times.npy']
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 537.6/537.6 kB 14.0 MB/s eta 0:00:00
Using device: cuda


## Cell 2 — Config

Same metadata-driven values as `ionosphere_s2cnn.ipynb` (grid, horizons, normalization all read from `metadata.json`, not hardcoded), so checkpoints from both notebooks are directly comparable in eval. No W&B here (dropped for now).

In [ ]:
with open("new_data/metadata.json") as f:
    _metadata = json.load(f)

_cadence_hours = _metadata['cadence_seconds'] / 3600
_n_horizons    = _metadata['target_steps']
HORIZON_NAMES  = [f"t+{int(_cadence_hours * (i + 1))}h" for i in range(_n_horizons)]

CONFIG = {
    # Grid -- derived from metadata, same convention as ionosphere_s2cnn.ipynb
    'H':     _metadata['nlat'],
    'W':     _metadata['nlon'],
    'l_max': _metadata['lmax'] + 1,   # metadata's lmax is raw degree; torch-harmonics wants count
    'm_max': _metadata['lmax'] + 1,

    # Data
    'n_timesteps':     _metadata['input_steps'],
    'n_omni_features': len(_metadata['omni_features']),
    'n_horizons':      _n_horizons,
    'horizon_names':   HORIZON_NAMES,
    'tec_mean':        _metadata['normalization']['tec_mean'],
    'tec_std':         _metadata['normalization']['tec_std'],
    'kp_feature_idx':  _metadata['omni_features'].index('kp_3hour'),
    'data_root':       'new_data',
    'flip_latitude':   False,  # confirmed via runtime check, twice, in two separate notebooks --
                               # data lats (ascending S->N) already match torch-harmonics' internal
                               # convention. See ionosphere_s2cnn.ipynb Cell 3 and this project's
                               # handoff doc if this ever needs re-verifying (new data, new grid, etc).

    # Model
    'gru_hidden_size': 128,
    'sfno_channels':   [64, 128, 128, 64],   # kept identical to SFNO for Flat CNN param-matching
    'n_conv_blocks':   4,

    # Training -- fewer epochs than SFNO's 20
    'batch_size':     64,
    'learning_rate':  1e-4,
    'n_epochs':       10,
    'n_kp_bins':      18,
}

print(f"Grid: {CONFIG['H']}x{CONFIG['W']} | l_max(count)={CONFIG['l_max']}")
print(f"Horizons: {CONFIG['horizon_names']}")
print(f"Baseline epochs: {CONFIG['n_epochs']} (SFNO uses 20)")

Grid: 23x45 | l_max(count)=23
Horizons: ['t+2h', 't+4h', 't+6h']
Baseline epochs: 10 (SFNO uses 20)


## Cell 3 — Dataset

In [ ]:
class FalishaDTECDataset(Dataset):
    """Inlined from data_pull/falisha_dataset.py (Hayden), reformatted 'new_data/' layout."""
    def __init__(self, root, split, flip_latitude=False):
        self.root = Path(root)
        self.split = split
        self.flip_latitude = flip_latitude

        self.tec_input  = np.load(self.root / f"{split}_tec_input.npy", mmap_mode="r")
        self.omni_input = np.load(self.root / f"{split}_omni_input.npy", mmap_mode="r")
        self.target      = np.load(self.root / f"{split}_target.npy", mmap_mode="r")
        self.window_start_times = np.load(self.root / f"{split}_window_start_times.npy", mmap_mode="r")

    def __len__(self):
        return int(self.tec_input.shape[0])

    def __getitem__(self, idx):
        tec_input = np.array(self.tec_input[idx], dtype=np.float32, copy=True)
        target    = np.array(self.target[idx], dtype=np.float32, copy=True)
        if self.flip_latitude:
            tec_input = np.flip(tec_input, axis=-2).copy()
            target    = np.flip(target, axis=-2).copy()
        return {
            "tec_input":  torch.from_numpy(tec_input),
            "omni_input": torch.from_numpy(np.array(self.omni_input[idx], dtype=np.float32, copy=True)),
            "target":     torch.from_numpy(target),
            "timestamp":  torch.tensor(int(self.window_start_times[idx]), dtype=torch.int64),
            "index":      torch.tensor(idx, dtype=torch.long),
        }


def make_dataloader(config, split, shuffle=None):
    if shuffle is None:
        shuffle = (split == 'train')
    ds = FalishaDTECDataset(config['data_root'], split, flip_latitude=config['flip_latitude'])
    return DataLoader(ds, batch_size=config['batch_size'], shuffle=shuffle)


_loader = make_dataloader(CONFIG, split='train')
_batch  = next(iter(_loader))
print('tec_input:', _batch['tec_input'].shape, '| omni_input:', _batch['omni_input'].shape,
      '| target:', _batch['target'].shape)

tec_input: torch.Size([64, 6, 23, 45]) | omni_input: torch.Size([64, 6, 6]) | target: torch.Size([64, 3, 23, 45])


## Cell 4 — Precompute: GL area weights, storm weight table

No Sobolev weights here -- that penalty is defined on SFNO's spectral filters specifically; neither baseline model has anything equivalent to apply it to.

In [ ]:
def make_gl_weights(H, device):
    _, w = legendre_gauss_weights(H)
    w    = w.to(torch.float32).to(device)
    return (w / w.sum()).view(1, 1, H, 1)


def build_storm_weight_table(kp_values, n_bins):
    kp_values = np.asarray(kp_values)
    bin_edges = np.linspace(kp_values.min(), kp_values.max(), n_bins + 1)
    counts, _ = np.histogram(kp_values, bins=bin_edges)
    density   = np.clip(counts / counts.sum(), 1e-8, None)
    weights   = (1.0 / density)
    weights   = weights / weights.mean()
    return torch.tensor(bin_edges, dtype=torch.float32), torch.tensor(weights, dtype=torch.float32)


def lookup_storm_weight(kp_batch, bin_edges, weights):
    bin_idx = torch.bucketize(kp_batch, bin_edges[1:-1])
    return weights.to(kp_batch.device)[bin_idx]


gl_weights = make_gl_weights(CONFIG['H'], DEVICE)

_train_ds_kp = FalishaDTECDataset(CONFIG['data_root'], 'train', CONFIG['flip_latitude'])
_train_kp    = np.array(_train_ds_kp.omni_input[:, -1, CONFIG['kp_feature_idx']])
storm_bin_edges, storm_weights_table = build_storm_weight_table(_train_kp, CONFIG['n_kp_bins'])
del _train_ds_kp, _train_kp

print(f'GL weights: {gl_weights.shape}')
print(f'Storm weight table: {CONFIG["n_kp_bins"]} bins, range '
      f'[{storm_weights_table.min():.2f}, {storm_weights_table.max():.2f}]')

GL weights: torch.Size([1, 1, 23, 1])
Storm weight table: 18 bins, range [0.01, 7.08]


## Cell 5 — Persistence + Climatology (zero-residual) baselines (no training)

Both training-free. Same `forward(tec_input, omni_input)` calling convention as the trained models, so everything shares one evaluation loop in `ionosphere_eval.ipynb`.

In [ ]:
class PersistenceBaseline:
    """Repeat last observed dTEC. Minimum meaningful bar."""
    def __call__(self, tec_input, omni_input):
        return tec_input[:, -1:].expand(-1, CONFIG['n_horizons'], -1, -1)


class ClimatologyBaseline:
    """Predicts zero residual (= predicts the climatological mean). NOT Hayden's real
    per-year absolute-TEC climatology .npz delivery -- that's a different thing entirely,
    used only in ionosphere_eval.ipynb to convert predictions to real TECU."""
    def __call__(self, tec_input, omni_input):
        B, T, H, W = tec_input.shape
        return torch.zeros(B, CONFIG['n_horizons'], H, W, device=tec_input.device)


print('Persistence + Climatology(zero-residual) baselines defined.')

Persistence + Climatology(zero-residual) baselines defined.


## Cell 6 — OMNI-only GRU

Predicts from solar wind alone. Thin wrapper keeps the same `forward(tec_input, omni_input)` signature (ignores `tec_input`) so one generic training loop works for both trained baselines.

In [ ]:
class OMNIOnlyGRU(nn.Module):
    def __init__(self, config):
        super().__init__()
        H, W         = config['H'], config['W']
        self.n_horiz = config['n_horizons']
        self.H, self.W = H, W
        self.gru  = nn.GRU(config['n_omni_features'], config['gru_hidden_size'], batch_first=True)
        self.head = nn.Linear(config['gru_hidden_size'], self.n_horiz * H * W)

    def forward(self, tec_input, omni_input):
        _, h_n = self.gru(omni_input)
        return self.head(h_n.squeeze(0)).view(-1, self.n_horiz, self.H, self.W)


_omni = OMNIOnlyGRU(CONFIG).to(DEVICE)
print(f'OMNIOnlyGRU parameters: {sum(p.numel() for p in _omni.parameters() if p.requires_grad):,}')
del _omni

OMNIOnlyGRU parameters: 452,769


## Cell 7 — Flat CNN (parameter-matched)

Full-map `Conv2d(kernel_size=(H, W))` in place of SFNO's SHT->filter->iSHT. Zero-padded in latitude, circularly padded in longitude (grid wraps). Same GRU encoder, ConvGRU-over-time structure, 4-block backbone/channel widths, output head as SFNO. Params: **51,182,211** vs. SFNO's **52,316,547** (2.2% difference) -- unaffected by the SFNO filter-shape fix, since this model never had an `l_max`/`m_max` filter to begin with.

In [ ]:
class FlatConvBlock(nn.Module):
    def __init__(self, in_channels, out_channels, H, W):
        super().__init__()
        self.H, self.W = H, W
        self.conv      = nn.Conv2d(in_channels, out_channels, kernel_size=(H, W))
        self.pointwise = nn.Conv2d(in_channels, out_channels, kernel_size=1)
        self.act       = nn.GELU()

    def _pad_full(self, x):
        x = F.pad(x, (self.W // 2, self.W - 1 - self.W // 2, 0, 0), mode='circular')
        x = F.pad(x, (0, 0, self.H // 2, self.H - 1 - self.H // 2), mode='constant', value=0)
        return x

    def forward(self, x):
        residual = self.pointwise(x)
        x_out    = self.conv(self._pad_full(x))
        return self.act(x_out + residual)


class FlatConvGRUCell(nn.Module):
    def __init__(self, input_channels, hidden_channels, H, W):
        super().__init__()
        combined = input_channels + hidden_channels
        self.reset_gate  = FlatConvBlock(combined, hidden_channels, H, W)
        self.update_gate = FlatConvBlock(combined, hidden_channels, H, W)
        self.candidate   = FlatConvBlock(combined, hidden_channels, H, W)

    def forward(self, x_t, h_prev):
        cat_xh  = torch.cat([x_t, h_prev], dim=1)
        r       = torch.sigmoid(self.reset_gate(cat_xh))
        z       = torch.sigmoid(self.update_gate(cat_xh))
        cat_xrh = torch.cat([x_t, r * h_prev], dim=1)
        h_cand  = torch.tanh(self.candidate(cat_xrh))
        return (1 - z) * h_prev + z * h_cand


class FlatCNNModel(nn.Module):
    def __init__(self, config):
        super().__init__()
        H, W     = config['H'], config['W']
        channels = config['sfno_channels']
        self.channels = channels

        self.omni_gru     = nn.GRU(config['n_omni_features'], config['gru_hidden_size'], batch_first=True)
        self.context_proj = nn.Linear(config['gru_hidden_size'], channels[0])
        self.conv_gru     = FlatConvGRUCell(1, channels[0], H, W)

        ins  = [channels[0], channels[1], channels[2], channels[3]]
        outs = [channels[1], channels[2], channels[3], channels[3]]
        self.conv_blocks = nn.ModuleList([
            FlatConvBlock(ins[i], outs[i], H, W) for i in range(config['n_conv_blocks'])
        ])
        self.output_head = nn.Conv2d(channels[-1], config['n_horizons'], kernel_size=1)

    def forward(self, tec_input, omni_input):
        B, T, H, W = tec_input.shape
        _, h_n  = self.omni_gru(omni_input)
        h_init  = self.context_proj(h_n.squeeze(0))
        h = h_init.unsqueeze(-1).unsqueeze(-1).expand(B, self.channels[0], H, W).contiguous()

        for t in range(T):
            h = self.conv_gru(tec_input[:, t].unsqueeze(1), h)

        x = h
        for block in self.conv_blocks:
            x = block(x)
        return self.output_head(x)


_flat = FlatCNNModel(CONFIG).to(DEVICE)
print(f'FlatCNNModel parameters: {sum(p.numel() for p in _flat.parameters() if p.requires_grad):,}  '
      f'(SFNO: 52,316,547)')
del _flat

FlatCNNModel parameters: 51,182,211  (SFNO: 52,316,547)


## Cell 8 — Loss, metrics, training/validation loop

Storm+area-weighted MSE, no Sobolev term (nothing to apply it to). One generic loop for both
models since both share `forward(tec_input, omni_input)`.

**Rewritten to use `tqdm` for per-batch progress** instead of periodic `print` statements --
gives a live ETA/it-per-sec instead of requiring batch-count x seconds/batch mental math to tell
if a run is still alive.


In [ ]:
def baseline_loss(pred, target, gl_weights, storm_weights):
    sq_err         = (pred - target) ** 2 * gl_weights
    mse_per_sample = sq_err.mean(dim=[1, 2, 3])
    w              = storm_weights / storm_weights.sum()
    return (w * mse_per_sample).sum()


def compute_metrics(pred, target, gl_weights, horizon_names):
    sq_err = (pred - target) ** 2 * gl_weights
    return {f'rmse_{name}': sq_err[:, i].mean().sqrt().item() for i, name in enumerate(horizon_names)}


def train_one_epoch(model, loader, optimizer, gl_weights, storm_bin_edges, storm_weights_table, config, device,
                     use_amp=False, scaler=None,
                     checkpoint_every=None, checkpoint_path=None, drive_checkpoint_path=None,
                     desc='train'):
    """
    use_amp/scaler: mixed precision. Pass a torch.amp.GradScaler('cuda') as `scaler` when
        use_amp=True.
    checkpoint_every: if set, save a checkpoint every N batches (not just at epoch end).
    Progress is shown via a live tqdm bar (updates every batch) -- watch it directly instead of
    inferring progress from print-interval timing.
    """
    model.train()
    total_loss = 0.0
    n_batches = len(loader)
    pbar = tqdm(loader, total=n_batches, desc=desc, unit='batch')

    for i, batch in enumerate(pbar):
        tec_input  = batch['tec_input'].to(device)
        omni_input = batch['omni_input'].to(device)
        target     = batch['target'].to(device)
        kp         = omni_input[:, -1, config['kp_feature_idx']].contiguous()
        sw         = lookup_storm_weight(kp, storm_bin_edges.to(device), storm_weights_table.to(device))

        optimizer.zero_grad()

        if use_amp:
            with torch.amp.autocast('cuda'):
                pred = model(tec_input, omni_input)
                loss = baseline_loss(pred, target, gl_weights, sw)
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)  # so clip_grad_norm_ operates on real-scale grads
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            scaler.step(optimizer)
            scaler.update()
        else:
            pred = model(tec_input, omni_input)
            loss = baseline_loss(pred, target, gl_weights, sw)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()

        batch_loss = loss.item()
        total_loss += batch_loss
        pbar.set_postfix(loss=f'{batch_loss:.4f}', avg=f'{total_loss / (i + 1):.4f}')

        if checkpoint_every is not None and checkpoint_path is not None and i > 0 and i % checkpoint_every == 0:
            save_checkpoint(model, optimizer, -1, total_loss / (i + 1), path=checkpoint_path)
            if drive_checkpoint_path is not None:
                shutil.copy(checkpoint_path, drive_checkpoint_path)
            pbar.write(f'  -> mid-epoch checkpoint saved at batch {i}/{n_batches}')

    return {'train/loss': total_loss / n_batches}


@torch.no_grad()
def validate_model(model, loader, gl_weights, storm_bin_edges, storm_weights_table, config, device, use_amp=False):
    model.eval()
    total_loss = 0.0
    rmse_accum = {name: 0.0 for name in config['horizon_names']}
    n = len(loader)
    pbar = tqdm(loader, total=n, desc='val', unit='batch')

    for batch in pbar:
        tec_input  = batch['tec_input'].to(device)
        omni_input = batch['omni_input'].to(device)
        target     = batch['target'].to(device)
        kp         = omni_input[:, -1, config['kp_feature_idx']].contiguous()
        sw         = lookup_storm_weight(kp, storm_bin_edges.to(device), storm_weights_table.to(device))

        if use_amp:
            with torch.amp.autocast('cuda'):
                pred = model(tec_input, omni_input)
                loss = baseline_loss(pred, target, gl_weights, sw)
        else:
            pred = model(tec_input, omni_input)
            loss = baseline_loss(pred, target, gl_weights, sw)
        total_loss += loss.item()
        pbar.set_postfix(loss=f'{loss.item():.4f}')

        m = compute_metrics(pred, target, gl_weights, config['horizon_names'])
        for name in rmse_accum:
            rmse_accum[name] += m[f'rmse_{name}']

    result = {'val/loss': total_loss / n}
    result.update({f'rmse_{name}': rmse_accum[name] / n for name in rmse_accum})
    return result


## Cell 9 — Checkpointing

In [ ]:
def save_checkpoint(model, optimizer, epoch, val_loss, path):
    torch.save({'epoch': epoch, 'model': model.state_dict(), 'optimizer': optimizer.state_dict(),
                'val_loss': val_loss, 'config': CONFIG}, path)
    print(f'  ✓ Saved checkpoint (epoch {epoch}, val_loss={val_loss:.4f}) -> {path}')

## Cell 10 — Build shared data loaders

In [ ]:
train_loader = make_dataloader(CONFIG, split='train')
val_loader   = make_dataloader(CONFIG, split='val')
print(f'train batches: {len(train_loader)}  |  val batches: {len(val_loader)}')

train batches: 1330  |  val batches: 193


## Cell 11 — Train OMNI-only GRU -> `best_omni_model.pt`

In [ ]:
omni_model     = OMNIOnlyGRU(CONFIG).to(DEVICE)
omni_optimizer = optim.Adam(omni_model.parameters(), lr=CONFIG['learning_rate'])
omni_scheduler = optim.lr_scheduler.ReduceLROnPlateau(omni_optimizer, mode='min', factor=0.5, patience=3)

print(f"OMNI-only GRU params: {sum(p.numel() for p in omni_model.parameters() if p.requires_grad):,}")
best_omni_val_loss = float('inf')

for epoch in range(CONFIG['n_epochs']):
    train_m = train_one_epoch(omni_model, train_loader, omni_optimizer, gl_weights,
                               storm_bin_edges, storm_weights_table, CONFIG, DEVICE)
    val_m = validate_model(omni_model, val_loader, gl_weights, storm_bin_edges, storm_weights_table, CONFIG, DEVICE)
    omni_scheduler.step(val_m['val/loss'])

    rmse_str = ' | '.join(f"rmse_{name}={val_m[f'rmse_{name}']:.4f}" for name in CONFIG['horizon_names'])
    print(f"Epoch {epoch:03d} | train={train_m['train/loss']:.4f} | val={val_m['val/loss']:.4f} | {rmse_str}")

    if val_m['val/loss'] < best_omni_val_loss:
        best_omni_val_loss = val_m['val/loss']
        save_checkpoint(omni_model, omni_optimizer, epoch, best_omni_val_loss, path='best_omni_model.pt')
        shutil.copy('best_omni_model.pt', '/content/drive/MyDrive/tec_data/best_omni_model.pt')
        print('  -> backed up to Drive')

print('Done with OMNI-only GRU.')

OMNI-only GRU params: 452,769


train:   0%|          | 0/1330 [00:00<?, ?batch/s]

val:   0%|          | 0/193 [00:00<?, ?batch/s]

Epoch 000 | train=0.0684 | val=0.0275 | rmse_t+2h=0.1538 | rmse_t+4h=0.1535 | rmse_t+6h=0.1532
  ✓ Saved checkpoint (epoch 0, val_loss=0.0275) -> best_omni_model.pt
  -> backed up to Drive


train:   0%|          | 0/1330 [00:00<?, ?batch/s]

...

val:   0%|          | 0/193 [00:00<?, ?batch/s]

Epoch 008 | train=0.0622 | val=0.0256 | rmse_t+2h=0.1442 | rmse_t+4h=0.1442 | rmse_t+6h=0.1442
  ✓ Saved checkpoint (epoch 8, val_loss=0.0256) -> best_omni_model.pt
  -> backed up to Drive


train:   0%|          | 0/1330 [00:00<?, ?batch/s]

val:   0%|          | 0/193 [00:00<?, ?batch/s]

Epoch 009 | train=0.0618 | val=0.0258 | rmse_t+2h=0.1446 | rmse_t+4h=0.1447 | rmse_t+6h=0.1451
Done with OMNI-only GRU.


## Cell 11.5 — Time one real Flat CNN batch before committing

Everything before this point has been estimates. This actually runs one forward+backward pass on
one real batch and times it directly, so `FLAT_N_EPOCHS`/`FLAT_TRAIN_FRACTION` below get set from
a measured number, not a guess.


In [ ]:
flat_model_probe     = FlatCNNModel(CONFIG).to(DEVICE)
flat_optimizer_probe = optim.Adam(flat_model_probe.parameters(), lr=CONFIG['learning_rate'])
flat_scaler_probe    = torch.amp.GradScaler('cuda')

probe_batch = next(iter(train_loader))
tec_input   = probe_batch['tec_input'].to(DEVICE)
omni_input  = probe_batch['omni_input'].to(DEVICE)
target      = probe_batch['target'].to(DEVICE)
kp          = omni_input[:, -1, CONFIG['kp_feature_idx']].contiguous()
sw          = lookup_storm_weight(kp, storm_bin_edges.to(DEVICE), storm_weights_table.to(DEVICE))

torch.cuda.synchronize()
t0 = time.time()

flat_optimizer_probe.zero_grad()
with torch.amp.autocast('cuda'):
    pred = flat_model_probe(tec_input, omni_input)
    loss = baseline_loss(pred, target, gl_weights, sw)
flat_scaler_probe.scale(loss).backward()
flat_scaler_probe.step(flat_optimizer_probe)
flat_scaler_probe.update()

torch.cuda.synchronize()
measured_sec_per_batch = time.time() - t0

print(f'Measured: {measured_sec_per_batch:.2f} s/batch (AMP on, batch_size={CONFIG["batch_size"]})')
print(f'Full epoch (1330 batches) would take: {1330 * measured_sec_per_batch / 60:.1f} min')
print(f'25% subsample (~332 batches) would take: {332 * measured_sec_per_batch / 60:.1f} min')

del flat_model_probe, flat_optimizer_probe, flat_scaler_probe
torch.cuda.empty_cache()


Measured: 7.59 s/batch (AMP on, batch_size=64)
Full epoch (1330 batches) would take: 168.3 min
25% subsample (~332 batches) would take: 42.0 min


## Cell 12 — Train Flat CNN -> `best_flat_model.pt`

**Why this is slow at all:** `FlatCNNModel` runs 22 full-grid `Conv2d(H,W)` convolutions per
forward pass -- 18 inside a `ConvGRU` loop over 6 TEC timesteps (3 gates x 6 steps, forced
sequential, can't be batched across time) plus 4 in the backbone. This is architectural, by
design (matches SFNO's global receptive field for a fair comparison), not a bug -- so the fix
isn't "make the model different," it's "fit the real cost into the time budget available."

**What this cell does about it:**
- **AMP** (`torch.amp.autocast('cuda')` + `GradScaler`) -- speeds up the raw conv/matmul math via
  tensor cores. Doesn't change *what's* computed, just how fast the arithmetic runs.
- **Per-epoch random subsample of train data** (`FLAT_TRAIN_FRACTION`) -- freshly resampled each
  epoch (different seed), so across several epochs the model still sees a broad spread of the
  data, just not all of it in any single epoch. Real tradeoff: noisier gradient signal per epoch
  than a full pass -- acceptable for a baseline, not free.
- **Mid-epoch checkpointing** every `CHECKPOINT_EVERY` batches, backed up to Drive immediately --
  so a disconnect mid-epoch doesn't cost you the whole epoch's progress.
- **Live `tqdm` progress bar** (from the updated Cell 8) -- updates every batch with running loss
  and time estimates. No more inferring "is it stuck" from print-interval math.
- Validation runs on the **full** val set (not subsampled) each epoch, since it's what decides
  which checkpoint gets kept as "best."

`FLAT_N_EPOCHS` and `FLAT_TRAIN_FRACTION` below are set using the number the timing cell above
just measured, not a guess -- re-run that cell if anything about the data/model changes and
re-derive these.


In [ ]:
# Set from the Cell 11.5 timing measurement above -- re-derive if that number changes.
FLAT_TRAIN_FRACTION = 0.25
FLAT_N_EPOCHS        = 6
CHECKPOINT_EVERY     = 50   # roughly every ~15% of an epoch at 332 batches/epoch

print(f'measured_sec_per_batch={measured_sec_per_batch:.2f}  ->  FLAT_N_EPOCHS={FLAT_N_EPOCHS}')

def make_subsampled_train_loader(config, fraction, seed):
    """Fresh random subset of the training set, different each epoch (seed=epoch), so the model
    sees a different slice of data each pass rather than the same reduced set every time."""
    full_ds = FalishaDTECDataset(config['data_root'], split='train', flip_latitude=config['flip_latitude'])
    n = len(full_ds)
    k = max(1, int(n * fraction))
    rng = np.random.default_rng(seed)
    idx = rng.choice(n, size=k, replace=False)
    subset = torch.utils.data.Subset(full_ds, idx.tolist())
    return DataLoader(subset, batch_size=config['batch_size'], shuffle=True)

flat_model     = FlatCNNModel(CONFIG).to(DEVICE)
flat_optimizer = optim.Adam(flat_model.parameters(), lr=CONFIG['learning_rate'])
flat_scheduler = optim.lr_scheduler.ReduceLROnPlateau(flat_optimizer, mode='min', factor=0.5, patience=3)
flat_scaler    = torch.amp.GradScaler('cuda')

print(f"Flat CNN params: {sum(p.numel() for p in flat_model.parameters() if p.requires_grad):,}  (SFNO: 52,316,547)")
best_flat_val_loss = float('inf')

for epoch in range(FLAT_N_EPOCHS):
    flat_train_loader = make_subsampled_train_loader(CONFIG, FLAT_TRAIN_FRACTION, seed=epoch)
    print(f"Epoch {epoch:03d} | training on {len(flat_train_loader.dataset):,} samples "
          f"({len(flat_train_loader)} batches)")

    train_m = train_one_epoch(
        flat_model, flat_train_loader, flat_optimizer, gl_weights,
        storm_bin_edges, storm_weights_table, CONFIG, DEVICE,
        use_amp=True, scaler=flat_scaler,
        checkpoint_every=CHECKPOINT_EVERY,
        checkpoint_path='flat_model_midepoch.pt',
        drive_checkpoint_path='/content/drive/MyDrive/tec_data/flat_model_midepoch.pt',
        desc=f'flat train e{epoch}',
    )
    val_m = validate_model(flat_model, val_loader, gl_weights, storm_bin_edges, storm_weights_table,
                            CONFIG, DEVICE, use_amp=True)
    flat_scheduler.step(val_m['val/loss'])

    rmse_str = ' | '.join(f"rmse_{name}={val_m[f'rmse_{name}']:.4f}" for name in CONFIG['horizon_names'])
    print(f"Epoch {epoch:03d} | train={train_m['train/loss']:.4f} | val={val_m['val/loss']:.4f} | {rmse_str}")

    if val_m['val/loss'] < best_flat_val_loss:
        best_flat_val_loss = val_m['val/loss']
        save_checkpoint(flat_model, flat_optimizer, epoch, best_flat_val_loss, path='best_flat_model.pt')
        shutil.copy('best_flat_model.pt', '/content/drive/MyDrive/tec_data/best_flat_model.pt')
        print('  -> backed up to Drive')

print('Done with Flat CNN.')


measured_sec_per_batch=7.59  ->  FLAT_N_EPOCHS=6
Flat CNN params: 51,182,211  (SFNO: 52,316,547)
Epoch 000 | training on 21,271 samples (333 batches)


flat train e0:   0%|          | 0/333 [00:00<?, ?batch/s]

  ✓ Saved checkpoint (epoch -1, val_loss=0.0837) -> flat_model_midepoch.pt
  -> mid-epoch checkpoint saved at batch 50/333
  ✓ Saved checkpoint (epoch -1, val_loss=0.0834) -> flat_model_midepoch.pt
  -> mid-epoch checkpoint saved at batch 100/333
  ✓ Saved checkpoint (epoch -1, val_loss=0.0810) -> flat_model_midepoch.pt
  -> mid-epoch checkpoint saved at batch 150/333
  ✓ Saved checkpoint (epoch -1, val_loss=0.0802) -> flat_model_midepoch.pt
  -> mid-epoch checkpoint saved at batch 200/333
  ✓ Saved checkpoint (epoch -1, val_loss=0.0786) -> flat_model_midepoch.pt
  -> mid-epoch checkpoint saved at batch 250/333
  ✓ Saved checkpoint (epoch -1, val_loss=0.0780) -> flat_model_midepoch.pt
  -> mid-epoch checkpoint saved at batch 300/333


val:   0%|          | 0/193 [00:00<?, ?batch/s]

Epoch 000 | train=0.0776 | val=0.0287 | rmse_t+2h=0.1509 | rmse_t+4h=0.1506 | rmse_t+6h=0.1595
  ✓ Saved checkpoint (epoch 0, val_loss=0.0287) -> best_flat_model.pt
  -> backed up to Drive
Epoch 001 | training on 21,271 samples (333 batches)


...

Epoch 004 | train=0.0757 | val=0.0292 | rmse_t+2h=0.1550 | rmse_t+4h=0.1539 | rmse_t+6h=0.1584
Epoch 005 | training on 21,271 samples (333 batches)


flat train e5:   0%|          | 0/333 [00:00<?, ?batch/s]

  ✓ Saved checkpoint (epoch -1, val_loss=0.0697) -> flat_model_midepoch.pt
  -> mid-epoch checkpoint saved at batch 50/333
  ✓ Saved checkpoint (epoch -1, val_loss=0.0737) -> flat_model_midepoch.pt
  -> mid-epoch checkpoint saved at batch 100/333
  ✓ Saved checkpoint (epoch -1, val_loss=0.0767) -> flat_model_midepoch.pt
  -> mid-epoch checkpoint saved at batch 150/333
  ✓ Saved checkpoint (epoch -1, val_loss=0.0779) -> flat_model_midepoch.pt
  -> mid-epoch checkpoint saved at batch 200/333
  ✓ Saved checkpoint (epoch -1, val_loss=0.0766) -> flat_model_midepoch.pt
  -> mid-epoch checkpoint saved at batch 250/333
  ✓ Saved checkpoint (epoch -1, val_loss=0.0781) -> flat_model_midepoch.pt
  -> mid-epoch checkpoint saved at batch 300/333


val:   0%|          | 0/193 [00:00<?, ?batch/s]

Epoch 005 | train=0.0772 | val=0.0292 | rmse_t+2h=0.1554 | rmse_t+4h=0.1542 | rmse_t+6h=0.1583
Done with Flat CNN.
